In [1]:
%cd ..

/Users/faizanfaisal/Laptop_Data/Davis_Courses/SE/project/notjs/Not.js


In [2]:
import pandas as pd
import os
import json
from pathlib import Path


## Combining all feature files for each website

In [3]:
base_dir = Path("output")

dfs = []

for website_dir in base_dir.iterdir():
    if website_dir.is_dir():
        file_path = website_dir / "features.xlsx"
        if file_path.exists():
            df = pd.read_excel(file_path)
            df["website"] = website_dir.name  # optional: keep track of source
            dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

print(combined_df)

      Unnamed: 0                                        script_name  \
0             18                 https://www.hp.com/us-en/home.html   
1             20                 https://www.hp.com/us-en/home.html   
2             22  https://www.googletagmanager.com/gtm.js?id=GTM...   
3             24  https://www.googletagmanager.com/gtm.js?id=GTM...   
4             26  https://www.googletagmanager.com/gtm.js?id=GTM...   
...          ...                                                ...   
6013       24847  https://px.mountain.com/st?ga_tracking_id=G-LX...   
6014       24855  https://px.mountain.com/st?ga_tracking_id=G-LX...   
6015       24859                         https://gs.mountain.com/gs   
6016       24863                         https://gs.mountain.com/gs   
6017       24867  https://px.mountain.com/st?ga_tracking_id=G-LX...   

            method_name  label  is_mixed  num_req_sent  num_nodes  num_edges  \
0               @309@63      0         0             0        495  

## Keeping only JS files

In [4]:
js_df = combined_df[combined_df["script_name"].str.contains(r"\.js", na=False)]
js_df

,Unnamed: 0,script_name,method_name,label,is_mixed,num_req_sent,num_nodes,num_edges,nodes_div_by_edges,edges_div_by_nodes,...,removeEventListener,sendBeacon,has_fingerprinting_eventlistner,descendant_of_fingerprinting,ascendant_of_fingerprinting,num_local,num_closure,num_global,num_script,website
2,22,https://www.googletagmanager.com/gtm.js?id=GTM...,ms@483@53,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,5,0,0,0,0,hp.com
3,24,https://www.googletagmanager.com/gtm.js?id=GTM...,ks@476@1152,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,5,0,0,0,0,hp.com
4,26,https://www.googletagmanager.com/gtm.js?id=GTM...,js@476@1022,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,5,0,0,0,0,hp.com
5,28,https://www.googletagmanager.com/gtm.js?id=GTM...,@824@229,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,5,0,0,0,0,hp.com
6,30,https://www.googletagmanager.com/gtm.js?id=GTM...,@826@237,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,5,0,0,0,0,hp.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6000,24095,https://bat.bing.com/bat.js,UET._push@0@18667,1,0,4,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,0,0,0,hubspot.com
6001,24623,https://bat.bing.com/bat.js,UET.fireBeacon@0@43326,1,0,2,3925,17820,0.220258,4.540127,...,0,0,0,27,0,0,0,0,0,hubspot.com
6002,24639,https://bat.bing.com/bat.js,UET._push@0@16964,1,0,2,3925,17820,0.220258,4.540127,...,0,0,0,27,0,0,0,0,0,hubspot.com
6003,24703,https://bat.bing.com/bat.js,UET.checkuetHostdocumentload@0@7903,1,0,2,3925,17820,0.220258,4.540127,...,0,2,0,27,0,0,0,0,0,hubspot.com


## Adding the JS filepaths in which the method occurs

In [5]:
# cache for already loaded json mappings
request_maps = {}

def get_js_path(row):
    website = row["website"]
    script = row["script_name"]

    json_path = f"./output/{website}/request_id.json"

    # load mapping once per website
    if website not in request_maps:
        if os.path.exists(json_path):
            with open(json_path, "r") as f:
                request_maps[website] = json.load(f)
        else:
            request_maps[website] = {}

    mapping = request_maps[website]

    # get request id
    req_id = mapping.get(script)

    if req_id is None:
        return None

    return f"./output/{website}/response/{req_id}.txt"


# create new column
js_df["js_file_path"] = js_df.apply(get_js_path, axis=1)

In [6]:
js_df = js_df[js_df.js_file_path.notna()]
df_with_js_file_path = js_df[js_df["method_name"].str.match(r"^[^@]+@")]
df_with_js_file_path

,Unnamed: 0,script_name,method_name,label,is_mixed,num_req_sent,num_nodes,num_edges,nodes_div_by_edges,edges_div_by_nodes,...,sendBeacon,has_fingerprinting_eventlistner,descendant_of_fingerprinting,ascendant_of_fingerprinting,num_local,num_closure,num_global,num_script,website,js_file_path
2,22,https://www.googletagmanager.com/gtm.js?id=GTM...,ms@483@53,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,5,0,0,0,0,hp.com,./output/hp.com/response/5232.12.txt
3,24,https://www.googletagmanager.com/gtm.js?id=GTM...,ks@476@1152,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,5,0,0,0,0,hp.com,./output/hp.com/response/5232.12.txt
4,26,https://www.googletagmanager.com/gtm.js?id=GTM...,js@476@1022,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,5,0,0,0,0,hp.com,./output/hp.com/response/5232.12.txt
9,36,https://cdn.dynamicyield.com/api/8784964/api_s...,p@1@254648,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,0,0,0,0,hp.com,./output/hp.com/response/5232.50.txt
10,38,https://cdn.dynamicyield.com/api/8784964/api_s...,new s@1@255620,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,0,0,0,0,hp.com,./output/hp.com/response/5232.50.txt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6000,24095,https://bat.bing.com/bat.js,UET._push@0@18667,1,0,4,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt
6001,24623,https://bat.bing.com/bat.js,UET.fireBeacon@0@43326,1,0,2,3925,17820,0.220258,4.540127,...,0,0,27,0,0,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt
6002,24639,https://bat.bing.com/bat.js,UET._push@0@16964,1,0,2,3925,17820,0.220258,4.540127,...,0,0,27,0,0,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt
6003,24703,https://bat.bing.com/bat.js,UET.checkuetHostdocumentload@0@7903,1,0,2,3925,17820,0.220258,4.540127,...,2,0,27,0,0,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt


## Parsing out the method calls and definition from the JS files

In [7]:
import re
from surrogate_2.replace_function_call import extract_call_snippet
from surrogate_2.balance_parentheses import (
    find_ending_index,
    find_matching_brace,
    find_function_definition,
)

CALL_MATCH_LIMIT = 10
_script_cache = {}
_method_call_cache = {}
_method_definition_cache = {}


def parse_method_name(method_name):
    parts = str(method_name).rsplit("@", 2)
    if len(parts) != 3:
        return None, None, None

    function_name, line_str, column_str = parts
    try:
        return function_name.strip(), int(line_str), int(column_str)
    except ValueError:
        return function_name.strip(), None, None


def load_script_contents(script_path):
    script_path = str(script_path)
    if script_path not in _script_cache:
        path = Path(script_path)
        if not path.exists():
            _script_cache[script_path] = (None, None)
        else:
            code = path.read_text(encoding="utf-8", errors="replace")
            code_lines = code.splitlines(keepends=True)
            _script_cache[script_path] = (code, code_lines)
    return _script_cache[script_path]


def unique_non_empty(items):
    seen = set()
    out = []
    for item in items:
        if not item:
            continue
        cleaned = item.strip()
        if not cleaned or cleaned in seen:
            continue
        seen.add(cleaned)
        out.append(cleaned)
    return out


def get_name_candidates(function_name):
    function_name = (function_name or "").strip()
    if not function_name:
        return []

    candidates = [function_name]
    last_segment = function_name.split(".")[-1].strip()
    if last_segment and last_segment not in candidates:
        candidates.append(last_segment)
    return candidates


def call_text_mentions_name(call_text, function_name):
    if not call_text:
        return False

    for candidate in get_name_candidates(function_name):
        if re.search(rf"(?<![\\w$]){re.escape(candidate)}\\s*\\(", call_text):
            return True
        if "." not in candidate and re.search(rf"\\.{re.escape(candidate)}\\s*\\(", call_text):
            return True
    return False


def is_probable_definition(code, start_idx, end_idx):
    prefix = code[max(0, start_idx - 40):start_idx]
    suffix = code[end_idx + 1:end_idx + 20].lstrip()

    if re.search(r"\bfunction\s+$", prefix):
        return True

    if suffix.startswith("{"):
        stripped_prefix = prefix.rstrip()
        if not stripped_prefix or stripped_prefix[-1] in "{,;:\n":
            return True

    return False


def extract_call_from_match(code, match_start):
    open_paren = code.find("(", match_start)
    if open_paren == -1:
        return None

    end_idx = find_ending_index(code, open_paren)
    if end_idx == -1:
        return None

    if is_probable_definition(code, match_start, end_idx):
        return None

    return code[match_start:end_idx + 1].strip()


def collect_method_calls(script_path, method_name):
    cache_key = (str(script_path), str(method_name))
    if cache_key in _method_call_cache:
        return _method_call_cache[cache_key]

    function_name, line_0, column_0 = parse_method_name(method_name)
    if not function_name or pd.isna(script_path):
        _method_call_cache[cache_key] = None
        return None

    code, code_lines = load_script_contents(script_path)
    if code is None or code_lines is None:
        _method_call_cache[cache_key] = None
        return None

    matches = []

    # Prefer the concrete runtime call site when it looks consistent with the method name.
    if line_0 is not None and column_0 is not None:
        try:
            exact_call, _ = extract_call_snippet(code_lines, line_0 + 1, column_0 + 1)
            if call_text_mentions_name(exact_call, function_name):
                matches.append(exact_call)
        except Exception:
            pass

    for candidate in get_name_candidates(function_name):
        patterns = [rf"(?<![\w$]){re.escape(candidate)}\s*\("]
        if "." not in candidate and len(candidate) >= 2:
            patterns.append(
                rf"(?<![\w$])(?:[A-Za-z_$][\w$]*\.)+{re.escape(candidate)}\s*\("
            )

        for pattern in patterns:
            for match in re.finditer(pattern, code):
                call_text = extract_call_from_match(code, match.start())
                if call_text:
                    matches.append(call_text)

                unique_matches = unique_non_empty(matches)
                if len(unique_matches) >= CALL_MATCH_LIMIT:
                    _method_call_cache[cache_key] = unique_matches
                    return unique_matches

    unique_matches = unique_non_empty(matches)
    result = unique_matches or None
    _method_call_cache[cache_key] = result
    return result


def extract_definition_from_match(code, match_start):
    paren_start = code.find("(", match_start)
    if paren_start == -1:
        return None

    paren_end = find_ending_index(code, paren_start)
    if paren_end == -1:
        return None

    idx = paren_end + 1
    while idx < len(code) and code[idx] in " \t\n\r":
        idx += 1

    if idx < len(code) and code[idx:idx + 2] == "=>":
        idx += 2
        while idx < len(code) and code[idx] in " \t\n\r":
            idx += 1

    if idx >= len(code):
        return code[match_start:paren_end + 1].strip()

    if code[idx] == "{":
        body_end = find_matching_brace(code, idx)
        if body_end != -1:
            return code[match_start:body_end + 1].strip()

    end_idx = idx
    while end_idx < len(code) and code[end_idx] not in ";\n":
        end_idx += 1
    return code[match_start:end_idx].strip()


def definition_starts_correctly(definition, function_name):
    if not definition:
        return False

    header = definition[:250]
    for candidate in get_name_candidates(function_name):
        escaped = re.escape(candidate)
        patterns = [
            rf"^\s*function\s+{escaped}\s*\(",
            rf"^\s*(?:const|let|var)?\s*{escaped}\s*=\s*function\s*\(",
            rf"^\s*(?:const|let|var)?\s*{escaped}\s*=\s*(?:async\s*)?(?:\([^)]*\)|[A-Za-z_$][\w$]*)\s*=>",
            rf"^\s*{escaped}\s*:\s*function\s*\(",
            rf"^\s*{escaped}\s*:\s*(?:async\s*)?(?:\([^)]*\)|[A-Za-z_$][\w$]*)\s*=>",
            rf"^\s*(?:[A-Za-z_$][\w$]*\.)+{escaped}\s*=\s*function\s*\(",
            rf"^\s*(?:[A-Za-z_$][\w$]*\.)+prototype\.{escaped}\s*=\s*function\s*\(",
            rf"^\s*(?:[A-Za-z_$][\w$]*\.)+{escaped}\s*=\s*(?:async\s*)?(?:\([^)]*\)|[A-Za-z_$][\w$]*)\s*=>",
        ]
        if any(re.search(pattern, header) for pattern in patterns):
            return True

    return False


def collect_method_definition(script_path, method_name):
    cache_key = (str(script_path), str(method_name))
    if cache_key in _method_definition_cache:
        return _method_definition_cache[cache_key]

    function_name, _, _ = parse_method_name(method_name)
    if not function_name or pd.isna(script_path):
        _method_definition_cache[cache_key] = None
        return None

    code, _ = load_script_contents(script_path)
    if code is None:
        _method_definition_cache[cache_key] = None
        return None

    direct_definition = find_function_definition(code, function_name)
    if definition_starts_correctly(direct_definition, function_name):
        _method_definition_cache[cache_key] = direct_definition
        return direct_definition

    for candidate in get_name_candidates(function_name):
        escaped = re.escape(candidate)
        patterns = [
            rf"\bfunction\s+{escaped}\s*\(",
            rf"(?<![\w$])(?:const|let|var)?\s*{escaped}\s*=\s*function\s*\(",
            rf"(?<![\w$])(?:const|let|var)?\s*{escaped}\s*=\s*(?:async\s*)?(?:\([^)]*\)|[A-Za-z_$][\w$]*)\s*=>",
            rf"(?<![\w$]){escaped}\s*:\s*function\s*\(",
            rf"(?<![\w$]){escaped}\s*:\s*(?:async\s*)?(?:\([^)]*\)|[A-Za-z_$][\w$]*)\s*=>",
            rf"(?<![\w$])(?:[A-Za-z_$][\w$]*\.)+{escaped}\s*=\s*function\s*\(",
            rf"(?<![\w$])(?:[A-Za-z_$][\w$]*\.)+prototype\.{escaped}\s*=\s*function\s*\(",
            rf"(?<![\w$])(?:[A-Za-z_$][\w$]*\.)+{escaped}\s*=\s*(?:async\s*)?(?:\([^)]*\)|[A-Za-z_$][\w$]*)\s*=>",
        ]

        for pattern in patterns:
            match = re.search(pattern, code, flags=re.MULTILINE)
            if not match:
                continue

            definition = extract_definition_from_match(code, match.start())
            if definition and definition_starts_correctly(definition, function_name):
                _method_definition_cache[cache_key] = definition
                return definition

    _method_definition_cache[cache_key] = None
    return None


def build_method_metadata(row):
    return pd.Series(
        {
            "method_calls_in_file": collect_method_calls(row["js_file_path"], row["method_name"]),
            "method_definition": collect_method_definition(row["js_file_path"], row["method_name"]),
        }
    )




In [ ]:
# df_with_js_file_path_2 = df_with_js_file_path.copy()
# df_with_js_file_path_2[["method_calls_in_file", "method_definition"]] = df_with_js_file_path_2.apply(
#     build_method_metadata,
#     axis=1,
# )
# df_with_js_file_path_2.to_csv("df_with_js_file_path_2.csv", index=False)


In [9]:
df_with_js_file_path_2 = pd.read_csv("df_with_js_file_path_2.csv")
df_with_js_file_path_2 

,Unnamed: 0,script_name,method_name,label,is_mixed,num_req_sent,num_nodes,num_edges,nodes_div_by_edges,edges_div_by_nodes,...,descendant_of_fingerprinting,ascendant_of_fingerprinting,num_local,num_closure,num_global,num_script,website,js_file_path,method_calls_in_file,method_definition
0,22,https://www.googletagmanager.com/gtm.js?id=GTM...,ms@483@53,0,0,0,495,2053,0.241111,4.147475,...,0,5,0,0,0,0,hp.com,./output/hp.com/response/5232.12.txt,"['ms(d)', 'ms(e)', 'ms(c.Nc)']",ms=function(a){return a&&hb(6)?(Array.isArray(...
1,24,https://www.googletagmanager.com/gtm.js?id=GTM...,ks@476@1152,0,0,0,495,2053,0.241111,4.147475,...,0,5,0,0,0,0,hp.com,./output/hp.com/response/5232.12.txt,"['ks(r)', 'ks(b,g,!1,d)', 'ks(a,void 0,void 0,...","function ks(a,b,c,d){try{gs(""3"");var e;return(..."
2,26,https://www.googletagmanager.com/gtm.js?id=GTM...,js@476@1022,0,0,0,495,2053,0.241111,4.147475,...,0,5,0,0,0,0,hp.com,./output/hp.com/response/5232.12.txt,"['js(f)', 'js(window)', 'js(w)']","function js(a){return a.origin!==""null""}"
3,36,https://cdn.dynamicyield.com/api/8784964/api_s...,p@1@254648,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,0,0,0,hp.com,./output/hp.com/response/5232.50.txt,"['p(t)', 'p(e)', 'p(""C"",void 0,void 0)', 'p(""E...","function p(e,t,r){var n=(0,c.T)(e)||t||r?{next..."
4,50,https://cdn.dynamicyield.com/api/8784964/api_s...,Object.get@1@145977,0,0,0,495,2053,0.241111,4.147475,...,0,5,0,0,0,0,hp.com,./output/hp.com/response/5232.50.txt,"['get(t)', 'get(e)', 'get(t,[i.eL.LOCAL])', 'g...","get=function(e,t){return T(e).dispatch(""get"",[..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
923,24095,https://bat.bing.com/bat.js,UET._push@0@18667,1,0,4,3925,17820,0.220258,4.540127,...,0,0,0,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt,"['_push(e.eventPushQueue[t])', '_push([e,t,o])...","_push=function(e){if(e[1]instanceof Array)if(""..."
924,24623,https://bat.bing.com/bat.js,UET.fireBeacon@0@43326,1,0,2,3925,17820,0.220258,4.540127,...,27,0,0,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt,"['fireBeacon(i)', 'this.fireBeacon(i)']",fireBeacon=function(e){for(var t=this.getClUrl...
925,24639,https://bat.bing.com/bat.js,UET._push@0@16964,1,0,2,3925,17820,0.220258,4.540127,...,27,0,0,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt,"['_push(e.eventPushQueue[t])', '_push([e,t,o])...","_push=function(e){if(e[1]instanceof Array)if(""..."
926,24703,https://bat.bing.com/bat.js,UET.checkuetHostdocumentload@0@7903,1,0,2,3925,17820,0.220258,4.540127,...,27,0,0,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt,"['checkuetHostdocumentload()', 'e.checkuetHost...",checkuetHostdocumentload=function(){var e=this...


In [10]:
valid_index = df_with_js_file_path_2[
    ["method_name", "method_calls_in_file", "method_definition"]
].dropna().index

df_with_js_file_path_2 = df_with_js_file_path_2.loc[valid_index]
df_with_js_file_path_2[["method_name", "method_calls_in_file", "method_definition"]]

,method_name,method_calls_in_file,method_definition
0,ms@483@53,"['ms(d)', 'ms(e)', 'ms(c.Nc)']",ms=function(a){return a&&hb(6)?(Array.isArray(...
1,ks@476@1152,"['ks(r)', 'ks(b,g,!1,d)', 'ks(a,void 0,void 0,...","function ks(a,b,c,d){try{gs(""3"");var e;return(..."
2,js@476@1022,"['js(f)', 'js(window)', 'js(w)']","function js(a){return a.origin!==""null""}"
3,p@1@254648,"['p(t)', 'p(e)', 'p(""C"",void 0,void 0)', 'p(""E...","function p(e,t,r){var n=(0,c.T)(e)||t||r?{next..."
4,Object.get@1@145977,"['get(t)', 'get(e)', 'get(t,[i.eL.LOCAL])', 'g...","get=function(e,t){return T(e).dispatch(""get"",[..."
...,...,...,...
923,UET._push@0@18667,"['_push(e.eventPushQueue[t])', '_push([e,t,o])...","_push=function(e){if(e[1]instanceof Array)if(""..."
924,UET.fireBeacon@0@43326,"['fireBeacon(i)', 'this.fireBeacon(i)']",fireBeacon=function(e){for(var t=this.getClUrl...
925,UET._push@0@16964,"['_push(e.eventPushQueue[t])', '_push([e,t,o])...","_push=function(e){if(e[1]instanceof Array)if(""..."
926,UET.checkuetHostdocumentload@0@7903,"['checkuetHostdocumentload()', 'e.checkuetHost...",checkuetHostdocumentload=function(){var e=this...


## Neutralizing method definitions using LLM

In [11]:
from tqdm import tqdm

In [12]:
# Use surrogate_2 to neutralize extracted method definitions and store results in temp_3
from surrogate_2.llm_neutralize import DEFAULT_MODEL, neutralize_tracking_call, get_fallback_replacement

def neutralize_method_code(row, use_llm: bool = True, model: str = DEFAULT_MODEL):
    method_definition = row.get("method_definition")
    if pd.isna(method_definition):
        return None

    method_definition = str(method_definition).strip()
    if not method_definition:
        return None

    context = method_definition[:2000]
    replacement = (
        neutralize_tracking_call(method_definition, context, model=model)
        if use_llm
        else None
    )
    return replacement or get_fallback_replacement(method_definition)

# Ensure tqdm works with pandas apply
tqdm.pandas()

# Apply neutralization (use_llm=False to skip API and use heuristic fallback only)
USE_LLM = True  # Set to False if OPENAI_API_KEY is not set
sample_df = df_with_js_file_path_2.copy()
temp_3 = sample_df.assign(
    method_code_neutralized=sample_df.apply(
        lambda row: neutralize_method_code(row, use_llm=USE_LLM),
        axis=1,
    )
)

In [13]:
print(temp_3.method_code_neutralized.iloc[0])

('ms=function(a){return a?Array.isArray(a)?a:[a]:[]}', '```json\n{\n  "replacementCode": "ms=function(a){return a?Array.isArray(a)?a:[a]:[]}",\n  "changesSummary": [\n    "Removed the call to hb(6) which likely checks for a tracking-related condition.",\n    "Removed the every function call that checks Im(b) and Gm(b), which are likely tracking-related functions.",\n    "Preserved the array handling logic to maintain non-tracking behavior."\n  ]\n}\n```')


In [14]:
temp_3.to_csv("temp_3.csv", index=False)

In [32]:
temp3 = pd.read_csv("temp_3.csv")

In [41]:
import ast
temp3["method_code_neutralized"] = temp3["method_code_neutralized"].apply(ast.literal_eval)

In [47]:
json.loads(temp_3.method_code_neutralized.iloc[5][1].strip("```json"))

{'replacementCode': 'function d(e){return null}',
 'changesSummary': ["Replaced the call to function 'p' with 'null' as it likely represents a tracking call, and its return value is not used for non-tracking purposes."]}

In [51]:
# Function to extract code and changes
def extract_parts(tup):
    code = tup[0]

    # Remove the ```json wrapping
    json_str = re.sub(r'^```json\s*|\s*```$', '', tup[1], flags=re.DOTALL)
    
    try:
        # Parse JSON
        parsed = json.loads(json_str)
        # Join all change summaries into a single string
        changes = " ".join(parsed.get("changesSummary", []))
    except:
        changes = ""
    
    return pd.Series([code, changes])

# Apply function to dataframe
temp_3[['method_code_refactored', 'changes_summary']] = temp3['method_code_neutralized'].apply(extract_parts)

# Show result
temp_3[['method_code_refactored', 'changes_summary']]

,method_code_refactored,changes_summary
0,ms=function(a){return a?Array.isArray(a)?a:[a]...,Removed the call to hb(6) which likely checks ...
1,"function ks(a,b,c,d){try{var e;return(e=ls(fun...",Removed calls to gs('3') and hs('3') as they a...
2,function js(a){return a.origin!==null},Replaced string comparison with null compariso...
3,"function p(e,t,r){var n=(0,c.T)(e)||t||r?{next...","Removed calls to n.subscribe, n.next, n.comple..."
4,"get=function(e,t){return null}",Replaced the dispatch call with null to remove...
...,...,...
923,"_push=function(e){if(e[1]instanceof Array)if(""...","Removed calls to firePidEvent, fireSendBeacon,..."
924,fireBeacon=function(e){for(var t=this.getClUrl...,Removed the call to navigator.sendBeacon and t...
925,"_push=function(e){if(e[1]instanceof Array)if(""...","Removed calls to firePidEvent, fireSendBeacon,..."
926,checkuetHostdocumentload=function(){var e=this...,Removed calls to e.dispatchq and e._push as th...


#### Removing incorrectly parsed

In [58]:
temp_4 = temp_3[temp_3['changes_summary'] != ""]

#### Saving final files

In [59]:
temp_4.to_csv("temp_4_with_changes.csv", index=False)

In [62]:
temp_4[["method_definition","method_code_refactored", "changes_summary"]].to_csv("refactored_code.csv", index=False)

In [64]:
pd.read_csv("refactored_code.csv")

,method_definition,method_code_refactored,changes_summary
0,ms=function(a){return a&&hb(6)?(Array.isArray(...,ms=function(a){return a?Array.isArray(a)?a:[a]...,Removed the call to hb(6) which likely checks ...
1,"function ks(a,b,c,d){try{gs(""3"");var e;return(...","function ks(a,b,c,d){try{var e;return(e=ls(fun...",Removed calls to gs('3') and hs('3') as they a...
2,"function js(a){return a.origin!==""null""}",function js(a){return a.origin!==null},Replaced string comparison with null compariso...
3,"function p(e,t,r){var n=(0,c.T)(e)||t||r?{next...","function p(e,t,r){var n=(0,c.T)(e)||t||r?{next...","Removed calls to n.subscribe, n.next, n.comple..."
4,"get=function(e,t){return T(e).dispatch(""get"",[...","get=function(e,t){return null}",Replaced the dispatch call with null to remove...
...,...,...,...
863,fireBeaconImg=function(e){if(!0!==this.uetConf...,fireBeaconImg=function(e){if(!0!==this.uetConf...,Removed the line setting the 'src' attribute o...
864,"_push=function(e){if(e[1]instanceof Array)if(""...","_push=function(e){if(e[1]instanceof Array)if(""...","Removed calls to firePidEvent, fireSendBeacon,..."
865,fireBeacon=function(e){for(var t=this.getClUrl...,fireBeacon=function(e){for(var t=this.getClUrl...,Removed the call to navigator.sendBeacon and t...
866,"_push=function(e){if(e[1]instanceof Array)if(""...","_push=function(e){if(e[1]instanceof Array)if(""...","Removed calls to firePidEvent, fireSendBeacon,..."


In [ ]:
pd.read_csv("temp_4_with_changes.csv")

,Unnamed: 0,script_name,method_name,label,is_mixed,num_req_sent,num_nodes,num_edges,nodes_div_by_edges,edges_div_by_nodes,...,num_closure,num_global,num_script,website,js_file_path,method_calls_in_file,method_definition,method_code_neutralized,method_code_refactored,changes_summary
0,22,https://www.googletagmanager.com/gtm.js?id=GTM...,ms@483@53,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,hp.com,./output/hp.com/response/5232.12.txt,"['ms(d)', 'ms(e)', 'ms(c.Nc)']",ms=function(a){return a&&hb(6)?(Array.isArray(...,('ms=function(a){return a?Array.isArray(a)?a:[...,ms=function(a){return a?Array.isArray(a)?a:[a]...,Removed the call to hb(6) which likely checks ...
1,24,https://www.googletagmanager.com/gtm.js?id=GTM...,ks@476@1152,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,hp.com,./output/hp.com/response/5232.12.txt,"['ks(r)', 'ks(b,g,!1,d)', 'ks(a,void 0,void 0,...","function ks(a,b,c,d){try{gs(""3"");var e;return(...","('function ks(a,b,c,d){try{var e;return(e=ls(f...","function ks(a,b,c,d){try{var e;return(e=ls(fun...",Removed calls to gs('3') and hs('3') as they a...
2,26,https://www.googletagmanager.com/gtm.js?id=GTM...,js@476@1022,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,hp.com,./output/hp.com/response/5232.12.txt,"['js(f)', 'js(window)', 'js(w)']","function js(a){return a.origin!==""null""}","('function js(a){return a.origin!==null}', '``...",function js(a){return a.origin!==null},Replaced string comparison with null compariso...
3,36,https://cdn.dynamicyield.com/api/8784964/api_s...,p@1@254648,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,hp.com,./output/hp.com/response/5232.50.txt,"['p(t)', 'p(e)', 'p(""C"",void 0,void 0)', 'p(""E...","function p(e,t,r){var n=(0,c.T)(e)||t||r?{next...","('function p(e,t,r){var n=(0,c.T)(e)||t||r?{ne...","function p(e,t,r){var n=(0,c.T)(e)||t||r?{next...","Removed calls to n.subscribe, n.next, n.comple..."
4,50,https://cdn.dynamicyield.com/api/8784964/api_s...,Object.get@1@145977,0,0,0,495,2053,0.241111,4.147475,...,0,0,0,hp.com,./output/hp.com/response/5232.50.txt,"['get(t)', 'get(e)', 'get(t,[i.eL.LOCAL])', 'g...","get=function(e,t){return T(e).dispatch(""get"",[...","('get=function(e,t){return null}', '```json\n{...","get=function(e,t){return null}",Replaced the dispatch call with null to remove...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
863,24055,https://bat.bing.com/bat.js,UET.fireBeaconImg@0@22141,1,0,6,3925,17820,0.220258,4.540127,...,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt,"['fireBeaconImg(o)', 'fireBeaconImg(n)', 'fire...",fireBeaconImg=function(e){if(!0!==this.uetConf...,('fireBeaconImg=function(e){if(!0!==this.uetCo...,fireBeaconImg=function(e){if(!0!==this.uetConf...,Removed the line setting the 'src' attribute o...
864,24095,https://bat.bing.com/bat.js,UET._push@0@18667,1,0,4,3925,17820,0.220258,4.540127,...,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt,"['_push(e.eventPushQueue[t])', '_push([e,t,o])...","_push=function(e){if(e[1]instanceof Array)if(""...",('_push=function(e){if(e[1]instanceof Array)if...,"_push=function(e){if(e[1]instanceof Array)if(""...","Removed calls to firePidEvent, fireSendBeacon,..."
865,24623,https://bat.bing.com/bat.js,UET.fireBeacon@0@43326,1,0,2,3925,17820,0.220258,4.540127,...,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt,"['fireBeacon(i)', 'this.fireBeacon(i)']",fireBeacon=function(e){for(var t=this.getClUrl...,('fireBeacon=function(e){for(var t=this.getClU...,fireBeacon=function(e){for(var t=this.getClUrl...,Removed the call to navigator.sendBeacon and t...
866,24639,https://bat.bing.com/bat.js,UET._push@0@16964,1,0,2,3925,17820,0.220258,4.540127,...,0,0,0,hubspot.com,./output/hubspot.com/response/3419.1259.txt,"['_push(e.eventPushQueue[t])', '_push([e,t,o])...","_push=function(e){if(e[1]instanceof Array)if(""...",('_push=function(e){if(e[1]instanceof Array)if...,"_push=function(e){if(e[1]instanceof Array)if(""...","Removed calls to firePidEvent, f

In [69]:
temp44 = pd.read_csv("temp_4_with_changes.csv")

In [72]:
vc = temp44[["js_file_path", "method_definition"]].value_counts()
num_duplicate_groups = (vc > 1).sum()

num_duplicate_groups

155

## Writing per-site `surrogate2` outputs

In [67]:
output_root = Path("./output")


def write_surrogate2_outputs(refactored_df: pd.DataFrame, output_root: Path = Path("./output")):
    written_files = []
    missing_matches = []

    website_dirs = sorted(path for path in output_root.iterdir() if path.is_dir())
    for website_dir in website_dirs:
        (website_dir / "surrogate2").mkdir(exist_ok=True)

    working_df = refactored_df.dropna(
        subset=["website", "js_file_path", "method_definition", "method_code_refactored"]
    ).copy()

    for js_file_path, file_rows in working_df.groupby("js_file_path", sort=False):
        source_path = Path(js_file_path)
        if not source_path.exists():
            missing_matches.append(
                {
                    "website": file_rows["website"].iloc[0],
                    "js_file_path": js_file_path,
                    "method_name": None,
                    "reason": "source file missing",
                }
            )
            continue

        file_text = source_path.read_text(encoding="utf-8", errors="ignore")
        original_text = file_text
        replacements_applied = 0

        ordered_rows = file_rows.assign(
            method_definition_length=file_rows["method_definition"].astype(str).str.len()
        ).sort_values("method_definition_length", ascending=False)

        for row in ordered_rows.itertuples(index=False):
            old_code = str(row.method_definition)
            new_code = str(row.method_code_refactored)

            if old_code in file_text:
                file_text = file_text.replace(old_code, new_code, 1)
                replacements_applied += 1
            else:
                missing_matches.append(
                    {
                        "website": row.website,
                        "js_file_path": row.js_file_path,
                        "method_name": row.method_name,
                        "reason": "method_definition not found in source file",
                    }
                )

        website = file_rows["website"].iloc[0]
        destination_path = output_root / website / "surrogate2" / source_path.name
        destination_path.write_text(file_text, encoding="utf-8")

        written_files.append(
            {
                "website": website,
                "source_path": str(source_path),
                "surrogate2_path": str(destination_path),
                "replacements_applied": replacements_applied,
                "file_changed": file_text != original_text,
            }
        )

    written_df = pd.DataFrame(written_files)
    if written_df.empty:
        summary_df = pd.DataFrame(
            columns=["website", "files_written", "replacements_applied", "files_changed"]
        )
    else:
        summary_df = (
            written_df.groupby("website", as_index=False)
            .agg(
                files_written=("surrogate2_path", "count"),
                replacements_applied=("replacements_applied", "sum"),
                files_changed=("file_changed", "sum"),
            )
            .sort_values("website")
        )

    missing_df = pd.DataFrame(missing_matches)
    return written_df, summary_df, missing_df


In [68]:
surrogate2_written_df, surrogate2_summary_df, surrogate2_missing_df = write_surrogate2_outputs(temp_4, output_root)
print(f"Wrote {len(surrogate2_written_df)} files across {surrogate2_summary_df['website'].nunique()} websites.")
print(f"Missing replacements: {len(surrogate2_missing_df)}")
surrogate2_summary_df

Wrote 45 files across 5 websites.
Missing replacements: 285


,website,files_written,replacements_applied,files_changed
0,booking.com,2,9,2
1,dell.com,7,145,7
2,hp.com,5,32,5
3,hubspot.com,18,244,18
4,shopify.com,13,153,13


## Improved `surrogate2` writing logic

In [73]:
def _normalize_code_snippet(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value or None



def _unique_values(series):
    values = []
    seen = set()
    for value in series:
        normalized = _normalize_code_snippet(value)
        if normalized is None or normalized in seen:
            continue
        values.append(normalized)
        seen.add(normalized)
    return values



def write_surrogate2_outputs(refactored_df: pd.DataFrame, output_root: Path = Path("./output")):
    written_files = []
    replacement_logs = []
    issue_logs = []

    website_dirs = sorted(path for path in output_root.iterdir() if path.is_dir())
    for website_dir in website_dirs:
        (website_dir / "surrogate2").mkdir(exist_ok=True)

    working_df = refactored_df.dropna(
        subset=["website", "js_file_path", "method_definition", "method_code_refactored"]
    ).copy()

    for column in ["js_file_path", "method_definition", "method_code_refactored", "method_name"]:
        if column in working_df.columns:
            working_df[column] = working_df[column].map(_normalize_code_snippet)

    working_df = working_df.dropna(
        subset=["js_file_path", "method_definition", "method_code_refactored"]
    ).copy()

    deduped_df = (
        working_df.groupby(["website", "js_file_path", "method_definition"], sort=False)
        .agg(
            method_names=("method_name", _unique_values),
            refactored_versions=("method_code_refactored", _unique_values),
            duplicate_rows=("method_name", "size"),
        )
        .reset_index()
    )

    deduped_df["method_code_refactored"] = deduped_df["refactored_versions"].str[0]
    deduped_df["method_definition_length"] = deduped_df["method_definition"].str.len()

    for js_file_path, file_rows in deduped_df.groupby("js_file_path", sort=False):
        source_path = Path(js_file_path)
        website = file_rows["website"].iloc[0]

        if not source_path.exists():
            issue_logs.append(
                {
                    "website": website,
                    "js_file_path": js_file_path,
                    "method_names": [],
                    "duplicate_rows": int(file_rows["duplicate_rows"].sum()),
                    "reason": "source file missing",
                }
            )
            continue

        original_text = source_path.read_text(encoding="utf-8", errors="ignore")
        file_text = original_text
        replacements_applied = 0
        duplicates_collapsed = 0
        overlap_or_duplicate_skips = 0
        unchanged_skips = 0
        conflicting_refactors = 0

        ordered_rows = file_rows.sort_values("method_definition_length", ascending=False)

        for row in ordered_rows.itertuples(index=False):
            old_code = row.method_definition
            refactored_versions = row.refactored_versions
            new_code = row.method_code_refactored
            duplicate_rows = int(row.duplicate_rows)
            duplicates_collapsed += max(duplicate_rows - 1, 0)
            method_names = row.method_names

            if len(refactored_versions) > 1:
                conflicting_refactors += 1
                issue_logs.append(
                    {
                        "website": website,
                        "js_file_path": js_file_path,
                        "method_names": method_names,
                        "duplicate_rows": duplicate_rows,
                        "reason": "multiple refactored versions for same method_definition; used first version",
                    }
                )

            if old_code == new_code:
                unchanged_skips += 1
                issue_logs.append(
                    {
                        "website": website,
                        "js_file_path": js_file_path,
                        "method_names": method_names,
                        "duplicate_rows": duplicate_rows,
                        "reason": "refactored code identical to original; skipped replacement",
                    }
                )
                continue

            if old_code in file_text:
                file_text = file_text.replace(old_code, new_code, 1)
                replacements_applied += 1
                replacement_logs.append(
                    {
                        "website": website,
                        "js_file_path": js_file_path,
                        "method_names": method_names,
                        "duplicate_rows": duplicate_rows,
                        "status": "replaced",
                    }
                )
            elif old_code in original_text:
                overlap_or_duplicate_skips += 1
                issue_logs.append(
                    {
                        "website": website,
                        "js_file_path": js_file_path,
                        "method_names": method_names,
                        "duplicate_rows": duplicate_rows,
                        "reason": "definition was present in original file but was already handled or overlapped by an earlier replacement",
                    }
                )
            else:
                issue_logs.append(
                    {
                        "website": website,
                        "js_file_path": js_file_path,
                        "method_names": method_names,
                        "duplicate_rows": duplicate_rows,
                        "reason": "method_definition not found in original source file",
                    }
                )

        destination_path = output_root / website / "surrogate2" / source_path.name
        destination_path.write_text(file_text, encoding="utf-8")

        written_files.append(
            {
                "website": website,
                "source_path": str(source_path),
                "surrogate2_path": str(destination_path),
                "candidate_rows": int(file_rows["duplicate_rows"].sum()),
                "unique_definitions": int(len(file_rows)),
                "duplicates_collapsed": duplicates_collapsed,
                "replacements_applied": replacements_applied,
                "overlap_or_duplicate_skips": overlap_or_duplicate_skips,
                "unchanged_skips": unchanged_skips,
                "conflicting_refactors": conflicting_refactors,
                "file_changed": file_text != original_text,
            }
        )

    written_df = pd.DataFrame(written_files)
    if written_df.empty:
        summary_df = pd.DataFrame(
            columns=[
                "website",
                "files_written",
                "candidate_rows",
                "unique_definitions",
                "duplicates_collapsed",
                "replacements_applied",
                "overlap_or_duplicate_skips",
                "unchanged_skips",
                "conflicting_refactors",
                "files_changed",
            ]
        )
    else:
        summary_df = (
            written_df.groupby("website", as_index=False)
            .agg(
                files_written=("surrogate2_path", "count"),
                candidate_rows=("candidate_rows", "sum"),
                unique_definitions=("unique_definitions", "sum"),
                duplicates_collapsed=("duplicates_collapsed", "sum"),
                replacements_applied=("replacements_applied", "sum"),
                overlap_or_duplicate_skips=("overlap_or_duplicate_skips", "sum"),
                unchanged_skips=("unchanged_skips", "sum"),
                conflicting_refactors=("conflicting_refactors", "sum"),
                files_changed=("file_changed", "sum"),
            )
            .sort_values("website")
        )

    replacement_log_df = pd.DataFrame(replacement_logs)
    issue_df = pd.DataFrame(issue_logs)
    return written_df, summary_df, replacement_log_df, issue_df


surrogate2_written_df, surrogate2_summary_df, surrogate2_replacement_log_df, surrogate2_issue_df = write_surrogate2_outputs(temp_4, output_root)
true_missing_count = int(
    (surrogate2_issue_df["reason"] == "method_definition not found in original source file").sum()
) if not surrogate2_issue_df.empty else 0

print(f"Wrote {len(surrogate2_written_df)} files across {surrogate2_summary_df['website'].nunique()} websites.")
print(f"True missing replacements: {true_missing_count}")
print(f"Logged issues/skips: {len(surrogate2_issue_df)}")
surrogate2_summary_df

Wrote 45 files across 5 websites.
True missing replacements: 0
Logged issues/skips: 110


,website,files_written,candidate_rows,unique_definitions,duplicates_collapsed,replacements_applied,overlap_or_duplicate_skips,unchanged_skips,conflicting_refactors,files_changed
0,booking.com,2,13,9,4,8,0,1,1,2
1,dell.com,7,243,150,93,132,16,2,16,7
2,hp.com,5,64,42,22,25,11,6,3,5
3,hubspot.com,18,347,246,101,228,6,12,13,18
4,shopify.com,13,201,151,50,137,2,12,9,13
